In [1]:
%pip install -q pgserver psycopg2-binary sqlalchemy pandas pyarrow ucimlrepo

import tempfile
import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

PG_DIR = tempfile.mkdtemp(prefix='hw3_pg_')
pg = pgserver.get_server(PG_DIR, cleanup_mode='stop')

uri = pg.get_uri().replace('postgresql://', 'postgresql+psycopg2://', 1)
engine = create_engine(uri, future=True)

with engine.connect() as conn:
    version = conn.execute(text('SELECT version()')).scalar_one()
    print(version)


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
PostgreSQL 16.2 on x86_64-pc-mingw64, compiled by gcc.exe (x86_64-posix-seh-rev2, Built by MinGW-W64 project) 12.2.0, 64-bit


In [2]:
from pathlib import Path
from ucimlrepo import fetch_ucirepo

Path('data').mkdir(exist_ok=True)

dataset = fetch_ucirepo(id=560)
df = pd.concat([dataset.data.features, dataset.data.targets], axis=1)

csv_path = 'data/hw3_dataset.csv'
df.to_csv(csv_path, index=False, encoding='utf-8')

print(df.shape)
print(df.columns.tolist())
print(df.head())

(8760, 14)
['Date', 'Rented Bike Count', 'Hour', 'Temperature', 'Humidity', 'Wind speed', 'Visibility', 'Dew point temperature', 'Solar Radiation', 'Rainfall', 'Snowfall', 'Seasons', 'Holiday', 'Functioning Day']
        Date  Rented Bike Count  Hour  Temperature  Humidity  Wind speed  \
0  1/12/2017                254     0         -5.2        37         2.2   
1  1/12/2017                204     1         -5.5        38         0.8   
2  1/12/2017                173     2         -6.0        39         1.0   
3  1/12/2017                107     3         -6.2        40         0.9   
4  1/12/2017                 78     4         -6.0        36         2.3   

   Visibility  Dew point temperature  Solar Radiation  Rainfall  Snowfall  \
0        2000                  -17.6              0.0       0.0       0.0   
1        2000                  -17.6              0.0       0.0       0.0   
2        2000                  -17.7              0.0       0.0       0.0   
3        2000         

In [3]:
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS hw3_bike_clean CASCADE"))
    conn.execute(text("DROP TABLE IF EXISTS hw3_bike_staging CASCADE"))
    conn.execute(text("""
        CREATE TABLE hw3_bike_staging (
            date_raw                   TEXT,
            rented_bike_count_raw      TEXT,
            hour_raw                   TEXT,
            temperature_raw            TEXT,
            humidity_raw               TEXT,
            wind_speed_raw             TEXT,
            visibility_raw             TEXT,
            dew_point_temperature_raw  TEXT,
            solar_radiation_raw        TEXT,
            rainfall_raw               TEXT,
            snowfall_raw               TEXT,
            seasons_raw                TEXT,
            holiday_raw                TEXT,
            functioning_day_raw        TEXT
        )
    """))
print('staging created')

staging created


In [4]:
raw_conn = engine.raw_connection()

try:
    with raw_conn.cursor() as cur:
        with open(csv_path, 'r', encoding='utf-8', newline='') as file:
            cur.copy_expert(
                """
                COPY hw3_bike_staging (
                    date_raw, rented_bike_count_raw, hour_raw, temperature_raw,
                    humidity_raw, wind_speed_raw, visibility_raw,
                    dew_point_temperature_raw, solar_radiation_raw,
                    rainfall_raw, snowfall_raw, seasons_raw,
                    holiday_raw, functioning_day_raw
                )
                FROM STDIN
                WITH (FORMAT csv, HEADER true, DELIMITER ',')
                """,
                file
            )
    raw_conn.commit()
finally:
    raw_conn.close()

pd.read_sql('SELECT COUNT(*) AS loaded_rows FROM hw3_bike_staging', engine)

,loaded_rows
0,8760


In [5]:
pd.read_sql("""
    SELECT 'holiday' AS col, holiday_raw AS value, COUNT(*) AS n
    FROM hw3_bike_staging GROUP BY holiday_raw
    UNION ALL
    SELECT 'functioning_day', functioning_day_raw, COUNT(*)
    FROM hw3_bike_staging GROUP BY functioning_day_raw
    ORDER BY col, value
""", engine)

,col,value,n
0,functioning_day,No,295
1,functioning_day,Yes,8465
2,holiday,Holiday,432
3,holiday,No Holiday,8328


In [6]:
pd.read_sql("""
    SELECT DISTINCT split_part(date_raw, '/', 1)::int AS first_part
    FROM hw3_bike_staging
    ORDER BY first_part
""", engine)

,first_part
0,1
1,2
2,3
3,4
4,5
5,6
6,7
7,8
8,9
9,10


In [7]:
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS hw3_bike_clean CASCADE"))
    conn.execute(text("""
        CREATE TABLE hw3_bike_clean (
            bike_row_id         BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
            rent_date           DATE         NOT NULL,
            rent_hour           SMALLINT     NOT NULL CHECK (rent_hour BETWEEN 0 AND 23),
            rented_bike_count   INTEGER      NOT NULL CHECK (rented_bike_count >= 0),
            temperature_c       NUMERIC(5,1),
            humidity_pct        SMALLINT     CHECK (humidity_pct BETWEEN 0 AND 100),
            wind_speed_ms       NUMERIC(4,1) CHECK (wind_speed_ms >= 0),
            visibility_10m      INTEGER      CHECK (visibility_10m >= 0),
            dew_point_c         NUMERIC(5,1),
            solar_radiation_mj  NUMERIC(4,2) CHECK (solar_radiation_mj >= 0),
            rainfall_mm         NUMERIC(5,1) CHECK (rainfall_mm >= 0),
            snowfall_cm         NUMERIC(5,1) CHECK (snowfall_cm >= 0),
            season              TEXT         NOT NULL
                                CHECK (season IN ('Winter','Spring','Summer','Autumn')),
            is_holiday          BOOLEAN      NOT NULL,
            is_functioning_day  BOOLEAN      NOT NULL
        )
    """))

    conn.execute(text("""
        INSERT INTO hw3_bike_clean (
            rent_date, rent_hour, rented_bike_count, temperature_c,
            humidity_pct, wind_speed_ms, visibility_10m, dew_point_c,
            solar_radiation_mj, rainfall_mm, snowfall_cm, season,
            is_holiday, is_functioning_day
        )
        SELECT
            to_date(NULLIF(TRIM(date_raw), ''), 'DD/MM/YYYY'),
            NULLIF(TRIM(hour_raw), '')::SMALLINT,
            NULLIF(TRIM(rented_bike_count_raw), '')::INTEGER,
            NULLIF(TRIM(temperature_raw), '')::NUMERIC(5,1),
            NULLIF(TRIM(humidity_raw), '')::SMALLINT,
            NULLIF(TRIM(wind_speed_raw), '')::NUMERIC(4,1),
            NULLIF(TRIM(visibility_raw), '')::INTEGER,
            NULLIF(TRIM(dew_point_temperature_raw), '')::NUMERIC(5,1),
            NULLIF(TRIM(solar_radiation_raw), '')::NUMERIC(4,2),
            NULLIF(TRIM(rainfall_raw), '')::NUMERIC(5,1),
            NULLIF(TRIM(snowfall_raw), '')::NUMERIC(5,1),
            NULLIF(TRIM(seasons_raw), ''),
            CASE TRIM(holiday_raw)
                WHEN 'Holiday'    THEN TRUE
                WHEN 'No Holiday' THEN FALSE
            END,
            CASE TRIM(functioning_day_raw)
                WHEN 'Yes' THEN TRUE
                WHEN 'No'  THEN FALSE
            END
        FROM hw3_bike_staging
    """))
print('clean table loaded')

clean table loaded


In [8]:
pd.read_sql("""
    SELECT
        (SELECT COUNT(*) FROM hw3_bike_staging)  AS staging_rows,
        COUNT(*)                                  AS clean_rows,
        MIN(rent_date)                            AS first_day,
        MAX(rent_date)                            AS last_day,
        COUNT(DISTINCT rent_date)                 AS distinct_days
    FROM hw3_bike_clean
""", engine)

,staging_rows,clean_rows,first_day,last_day,distinct_days
0,8760,8760,2017-12-01,2018-11-30,365



| Колонка | Семантика | Формат у джерелі | Тип у clean | Правила очищення |
|---|---|---|---|---|
| Date | календарна доба оренди | текст `D/M/YYYY` | `DATE NOT NULL` | `to_date(..., 'DD/MM/YYYY')`; час не потрібен, тому не `TIMESTAMPTZ` |
| Hour | година доби | ціле 0-23 | `SMALLINT`, `CHECK 0..23` | `NULLIF(TRIM(...), '')` |
| Rented Bike Count | кількість оренд за годину (ціль) | ціле | `INTEGER NOT NULL`, `CHECK >= 0` | кількість не буває від'ємною |
| Temperature | температура повітря, °C | десяткове | `NUMERIC(5,1)` | точність до десятих, як у джерелі |
| Humidity | вологість, % | ціле | `SMALLINT`, `CHECK 0..100` | відсоток поза межами 0-100 неможливий |
| Wind speed | швидкість вітру, м/с | десяткове | `NUMERIC(4,1)`, `CHECK >= 0` | |
| Visibility | видимість, одиниці по 10 м | ціле | `INTEGER`, `CHECK >= 0` | |
| Dew point temperature | точка роси, °C | десяткове | `NUMERIC(5,1)` | |
| Solar Radiation | сонячна радіація, МДж/м² | десяткове | `NUMERIC(4,2)`, `CHECK >= 0` | |
| Rainfall / Snowfall | опади: дощ (мм), сніг (см) | десяткове | `NUMERIC(5,1)`, `CHECK >= 0` | |
| Seasons | пора року | текст, 4 значення | `TEXT`, `CHECK IN (...)` | `TEXT` замість `VARCHAR(255)`, бо бізнес-ліміту немає |
| Holiday | святковий день | `Holiday` / `No Holiday` | `BOOLEAN NOT NULL` | `Holiday` → TRUE, `No Holiday` → FALSE |
| Functioning Day | чи працювала система | `Yes` / `No` | `BOOLEAN NOT NULL` | `Yes` → TRUE, `No` → FALSE |





In [9]:
def show(title, sql):
    print(title)
    display(pd.read_sql(text(sql), engine))

In [10]:
show("3.1 Кількість рядків", """
SELECT 'staging' AS layer, COUNT(*) AS n_rows FROM hw3_bike_staging
UNION ALL
SELECT 'clean', COUNT(*) FROM hw3_bike_clean
""")

3.1 Кількість рядків


,layer,n_rows
0,staging,8760
1,clean,8760


In [11]:
show("3.2 Дублікати (business key = дата + година)", """
SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT (rent_date, rent_hour)) AS distinct_business_keys,
       COUNT(*) - COUNT(DISTINCT (rent_date, rent_hour)) AS duplicate_rows
FROM hw3_bike_clean
""")

3.2 Дублікати (business key = дата + година)


,total_rows,distinct_business_keys,duplicate_rows
0,8760,8760,0


3.1. У staging і clean по 8760 рядків. Кількість однакова, бо під час перенесення я нічого не фільтрувала: усі значення пройшли приведення типів і CHECK-обмеження. 8760 це 365 діб × 24 години, тож набір повний.

3.2. Business key (дата, година) дав 8760 унікальних комбінацій, тобто 0 дублікатів. Це логічно для погодинних даних: у кожну годину кожної доби один рядок. Нуль дублікатів за цим ключем не доводить, що всі значення правильні: помилка могла б сидіти в самих числах, і це перевіряють наступні кроки.

In [12]:
cols = [c for c in pd.read_sql(text("SELECT * FROM hw3_bike_staging LIMIT 0"), engine).columns]
parts = ", ".join(
    f"COUNT(*) FILTER (WHERE {c} IS NULL OR LOWER(TRIM({c})) IN ('', '?', 'unknown', 'na', 'n/a', 'null', 'nan')) AS {c}_missing"
    for c in cols
)
show("3.3a Пропуски й маркери у staging", f"SELECT {parts} FROM hw3_bike_staging")

3.3a Пропуски й маркери у staging


,date_raw_missing,rented_bike_count_raw_missing,hour_raw_missing,temperature_raw_missing,humidity_raw_missing,wind_speed_raw_missing,visibility_raw_missing,dew_point_temperature_raw_missing,solar_radiation_raw_missing,rainfall_raw_missing,snowfall_raw_missing,seasons_raw_missing,holiday_raw_missing,functioning_day_raw_missing
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [13]:
show("3.3b Null-rate у clean", """
SELECT
  ROUND(COUNT(*) FILTER (WHERE temperature_c IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_temperature,
  ROUND(COUNT(*) FILTER (WHERE humidity_pct  IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_humidity,
  ROUND(COUNT(*) FILTER (WHERE wind_speed_ms IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_wind,
  ROUND(COUNT(*) FILTER (WHERE visibility_10m IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_visibility,
  ROUND(COUNT(*) FILTER (WHERE rainfall_mm   IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_rainfall,
  ROUND(COUNT(*) FILTER (WHERE snowfall_cm   IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_snowfall,
  COUNT(*) AS total_rows
FROM hw3_bike_clean
""")

3.3b Null-rate у clean


,pct_null_temperature,pct_null_humidity,pct_null_wind,pct_null_visibility,pct_null_rainfall,pct_null_snowfall,total_rows
0,0.0,0.0,0.0,0.0,0.0,0.0,8760


3.3. Аудит 14 сирих колонок не знайшов порожніх рядків, ?, unknown чи інших маркерів пропуску, а в clean NULL-rate для температури, вологості, вітру, видимості, дощу й снігу дорівнює 0%. Нульовий результат не гарантує повної якості: пропуск міг бути заповнений «нормальним» числом (наприклад, 0 для опадів), а значення можуть бути помилковими, хоч і непорожніми. Це перевіряють розподіли й outlier-check у наступному завданні.

In [14]:
show("4.1 Розподіл за порою року", """
SELECT season, COUNT(*) AS n_rows,
       ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_rows
FROM hw3_bike_clean
GROUP BY season
ORDER BY n_rows DESC, season
LIMIT 10
""")

4.1 Розподіл за порою року


,season,n_rows,pct_rows
0,Spring,2208,25.21
1,Summer,2208,25.21
2,Autumn,2184,24.93
3,Winter,2160,24.66


In [15]:
show("4.2 Статистика rented_bike_count", """
SELECT COUNT(rented_bike_count) AS n_non_null,
       MIN(rented_bike_count) AS min_value,
       MAX(rented_bike_count) AS max_value,
       ROUND(AVG(rented_bike_count)::NUMERIC, 2) AS avg_value,
       ROUND(STDDEV_SAMP(rented_bike_count)::NUMERIC, 2) AS sd_value
FROM hw3_bike_clean
""")

4.2 Статистика rented_bike_count


,n_non_null,min_value,max_value,avg_value,sd_value
0,8760,0,3556,704.6,645.0


4.1. Розподіл за порами року майже рівномірний: Spring і Summer по 2208 рядків (25,21%), Autumn 2184 (24,93%), Winter 2160 (24,66%). Різниця пояснюється довжиною пір: 92, 92, 91 і 90 діб × 24 години. Рідкісних або «дивних» категорій немає, тож дисбаланс класів тут не загрожує аналізу.

4.2. Кількість оренд за годину змінюється від 0 до 3556, середнє 704,6, стандартне відхилення 645,0. Відхилення майже дорівнює середньому, а максимум лежить далеко від нього (близько 4,4σ). Це ознака сильно скошеного розподілу з довгим правим хвостом: більшість годин мають помірний попит, а кілька піків значно вищі.

In [16]:
show("4.3a Статистичний outlier-check (3σ)", """
WITH stats AS (
    SELECT AVG(rented_bike_count) AS mean_value,
           STDDEV_SAMP(rented_bike_count) AS sd_value
    FROM hw3_bike_clean
)
SELECT t.rent_date, t.rent_hour, t.rented_bike_count,
       ROUND(((t.rented_bike_count - s.mean_value) / NULLIF(s.sd_value, 0))::NUMERIC, 2) AS z_score
FROM hw3_bike_clean AS t
CROSS JOIN stats AS s
WHERE s.sd_value > 0
  AND ABS(t.rented_bike_count - s.mean_value) > 3 * s.sd_value
ORDER BY ABS(t.rented_bike_count - s.mean_value) DESC
LIMIT 20
""")

4.3a Статистичний outlier-check (3σ)


,rent_date,rent_hour,rented_bike_count,z_score
0,2018-06-19,18,3556,4.42
1,2018-06-21,18,3418,4.21
2,2018-06-12,18,3404,4.19
3,2018-06-20,18,3384,4.15
4,2018-06-04,18,3380,4.15
5,2018-06-22,18,3365,4.12
6,2018-06-08,18,3309,4.04
7,2018-09-10,18,3298,4.02
8,2018-09-17,18,3277,3.99
9,2018-09-12,18,3256,3.96


4.3a. За правилом 3σ знайдено 20 найбільших значень (показано LIMIT 20), z-score від 3,81 до 4,42. Усі вони припадають на 18:00 у травні-червні та вересні, тобто на вечірню годину пік у комфортну погоду. Оскільки розподіл скошений вправо (див. 4.2), 3σ тут позначає не помилки, а природні піки попиту. Дія: залишити як валідні рідкісні випадки, без видалення. Для моделювання можна додати прапорець is_peak, щоб модель не сприймала їх як шум.

In [17]:
show("4.3b Доменний check: оренди в нефункціональні години", """
SELECT is_functioning_day,
       COUNT(*) AS n_hours,
       COUNT(*) FILTER (WHERE rented_bike_count > 0) AS hours_with_rentals,
       MAX(rented_bike_count) AS max_rentals
FROM hw3_bike_clean
GROUP BY is_functioning_day
""")

4.3b Доменний check: оренди в нефункціональні години


,is_functioning_day,n_hours,hours_with_rentals,max_rentals
0,False,295,0,0
1,True,8465,8465,3556


4.3b. У 295 нефункціональних годинах rented_bike_count дорівнює 0 в усіх випадках, а в 8465 функціональних годинах оренди є в кожній (мінімум більше 0, максимум 3556). Отже, Functioning Day майже повністю визначає, чи буде ціль нульовою. Це доменно логічно: коли система не працює, велосипеди не беруть. Дія: залишити рядки в таблиці, але для навчання моделі окремо розглянути виключення нефункціональних годин, а Functioning Day як feature використовувати лише якщо статус відомий заздалегідь (див. leakage в Reflection).

In [18]:
# Q1. Які 5 годин доби мають найбільший середній попит?
show("Q1", """
SELECT rent_hour, ROUND(AVG(rented_bike_count), 1) AS avg_rentals, COUNT(*) AS n_rows
FROM hw3_bike_clean
WHERE is_functioning_day
GROUP BY rent_hour
ORDER BY avg_rentals DESC, rent_hour
LIMIT 5
""")

Q1


,rent_hour,avg_rentals,n_rows
0,18,1554.0,353
1,19,1235.8,353
2,17,1177.2,353
3,20,1105.3,353
4,21,1066.5,353


In [19]:
# Q2. Як різниться попит за порами року?
show("Q2", """
SELECT season, COUNT(*) AS n_hours, SUM(rented_bike_count) AS total_rentals,
       ROUND(AVG(rented_bike_count), 1) AS avg_per_hour
FROM hw3_bike_clean
WHERE is_functioning_day
GROUP BY season
ORDER BY total_rentals DESC
""")

Q2


,season,n_hours,total_rentals,avg_per_hour
0,Summer,2208,2283234,1034.1
1,Autumn,1937,1790002,924.1
2,Spring,2160,1611909,746.3
3,Winter,2160,487169,225.5


In [20]:
# Q3. Чим різняться святкові й звичайні дні?
show("Q3", """
SELECT is_holiday, COUNT(DISTINCT rent_date) AS n_days, COUNT(*) AS n_hours,
       ROUND(AVG(rented_bike_count), 1) AS avg_rentals
FROM hw3_bike_clean
WHERE is_functioning_day
GROUP BY is_holiday
ORDER BY is_holiday
""")

Q3


,is_holiday,n_days,n_hours,avg_rentals
0,False,336,8057,739.3
1,True,17,408,529.2


Q1. Найвищий середній попит у годинах 18, 19, 17, 20 і 21, причому лідер 18:00 має в середньому 1554,0 оренди за годину, на третину більше за другу за величиною 19:00 (1235,8). Усі п'ять годин припадають на вечірній час після роботи, і в кожній по 353 рядки (діб із працюючою системою), тож порівняння чесне. Для прогнозування година доби буде однією з найсильніших ознак.

Q2. Літо лідирує за сумою (2 283 234) і за середнім (1034,1 за годину), зима найнижча: 487 169 і 225,5, тобто середній попит улітку приблизно в 4,6 раза вищий. Осінь має лише 1937 годин замість 2184 календарних: бракує 247 нефункціональних годин, а навесні 48 (усього 295, як у 4.3b). Тому порівнювати пори за сумою некоректно, і середнє за годину тут чесніше.

Q3. У звичайні дні середній попит 739,3 оренди за годину (336 діб), у святкові 529,2 (17 діб), тобто приблизно на 28% нижче. Святкових діб небагато, тож різниця може частково залежати від того, на яку пору року вони припали. Окрім того, групу «звичайні дні» входять і вихідні, бо прапорець Holiday позначає лише офіційні свята. Для ML-моделі це означає, що день тижня варто додати окремою ознакою.


In [21]:
# Q4. Коли в години пік, у помірну суху погоду, брали найбільше велосипедів?
show("Q4", """
SELECT rent_date, rent_hour, rented_bike_count, temperature_c, rainfall_mm
FROM hw3_bike_clean
WHERE rent_hour IN (7, 8, 9, 17, 18, 19)
  AND temperature_c BETWEEN 15 AND 30
  AND NOT (rainfall_mm > 0 OR snowfall_cm > 0)
ORDER BY rented_bike_count DESC, rent_date
LIMIT 10
""")

Q4


,rent_date,rent_hour,rented_bike_count,temperature_c,rainfall_mm
0,2018-06-19,18,3556,24.1,0.0
1,2018-06-21,18,3418,27.8,0.0
2,2018-06-12,18,3404,24.9,0.0
3,2018-06-20,18,3384,27.0,0.0
4,2018-06-04,18,3380,24.4,0.0
5,2018-06-22,18,3365,29.3,0.0
6,2018-06-08,18,3309,26.2,0.0
7,2018-09-10,18,3298,25.9,0.0
8,2018-09-17,18,3277,25.3,0.0
9,2018-09-12,18,3256,27.0,0.0


Q4. У години пік (7-9 та 17-19) у комфортну суху погоду (температура від 15 до 30 °C, без дощу й снігу) найбільший попит зафіксовано 19 червня 2018 о 18:00: 3556 оренд при 24,1 °C. Усі десять рядків припадають на 18:00, на червень і вересень, температура від 24,1 до 29,3 °C, опадів немає. Отже, абсолютні рекорди попиту це вечірній пік після роботи в теплу суху погоду, а не випадкові викиди.

In [22]:
# Q5. Який попит і температура у порах року, назва яких починається на S?
show("Q5", """
SELECT season, COUNT(*) AS n_hours,
       ROUND(AVG(temperature_c), 1) AS avg_temp,
       ROUND(AVG(rented_bike_count), 1) AS avg_rentals
FROM hw3_bike_clean
WHERE season ILIKE 's%'
GROUP BY season
ORDER BY season
""")

Q5


,season,n_hours,avg_temp,avg_rentals
0,Spring,2208,13.0,730.0
1,Summer,2208,26.6,1034.1


Q5. Назву на S мають лише дві пори року. Літо тепліше (середня температура 26,6 °C проти 13,0 °C навесні) і має вищий середній попит (1034,1 проти 730,0 оренди за годину). Це узгоджується з Q2, де літо теж лідирує. Різниця для весни між цим запитом і Q2 (730,0 проти 746,3) пояснюється тим, що тут не відфільтровано нефункціональні години з нульовим попитом. Фільтр за початковою літерою зручний для демонстрації ILIKE, але для аналізу точніше фільтрувати за переліком (IN).

In [23]:
# Q6. Яка частка годин з дощем і яка інтенсивність дощу у ці години?
show("Q6", """
SELECT COUNT(*) AS n_hours,
       COUNT(*) FILTER (WHERE rainfall_mm IS NULL) AS null_rainfall,
       COUNT(*) FILTER (WHERE rainfall_mm IS NOT NULL) AS known_rainfall,
       COUNT(NULLIF(rainfall_mm, 0)) AS rainy_hours,
       ROUND(COUNT(NULLIF(rainfall_mm, 0)) * 100.0 / COUNT(*), 2) AS pct_rainy,
       ROUND(AVG(NULLIF(rainfall_mm, 0)), 2) AS avg_mm_in_rainy_hours,
       COALESCE(MAX(rainfall_mm), 0) AS max_mm
FROM hw3_bike_clean
""")

Q6


,n_hours,null_rainfall,known_rainfall,rainy_hours,pct_rainy,avg_mm_in_rainy_hours,max_mm
0,8760,0,8760,528,6.03,2.47,35.0


Q6. Дощ ішов у 528 годинах із 8760, тобто у 6,03% часу. Пропусків у rainfall_mm немає (null_rainfall = 0, known_rainfall = 8760), тому IS NULL та COALESCE тут нічого не змінюють, але підтверджують повноту даних. У дощові години середня інтенсивність 2,47 мм, а максимум 35,0 мм за годину, тобто це рідкісна злива. Така сильна нерівність (більшість годин сухі, кілька дуже вологих) означає, що для моделі rainfall_mm краще доповнити бінарною ознакою «був дощ».

In [24]:
# Q7. Чи були години без жодної оренди, коли система працювала?
show("Q7", """
SELECT rent_date, rent_hour, rented_bike_count, temperature_c
FROM hw3_bike_clean
WHERE rented_bike_count = 0
  AND is_functioning_day IS DISTINCT FROM FALSE
ORDER BY rent_date, rent_hour
LIMIT 20
""")

Q7


,rent_date,rent_hour,rented_bike_count,temperature_c


Q7. Запит шукав години, коли оренд не було (rented_bike_count = 0), хоча система працювала. Для умови is_functioning_day IS DISTINCT FROM FALSE це означає, що рядок проходить, якщо прапорець TRUE або невідомий (NULL). Результат порожній: усі 295 годин із нульовим попитом припадають на нефункціональні години. Отже, нуль оренд у цьому наборі означає не «ніхто не захотів поїхати», а «система не працювала», і це треба враховувати при моделюванні. Оператор IS DISTINCT FROM тут безпечніший за <> FALSE: він не відкидає рядки з NULL, хоча в нашій таблиці пропусків немає.

In [25]:
# Q8. Які комбінації пори року й святкового прапорця існують?
show("Q8", """
SELECT DISTINCT season, is_holiday
FROM hw3_bike_clean
ORDER BY season, is_holiday
""")

Q8


,season,is_holiday
0,Autumn,False
1,Autumn,True
2,Spring,False
3,Spring,True
4,Summer,False
5,Summer,True
6,Winter,False
7,Winter,True


Q8. DISTINCT повернув 8 унікальних комбінацій: кожна з чотирьох пір року (Autumn, Spring, Summer, Winter) зустрічається і зі is_holiday = FALSE, і зі TRUE. Отже, у кожну пору є хоча б одна святкова доба, і порівняння святкових і звичайних днів усередині пори можливе. Запит показує лише наявність комбінацій, а не їх кількість: святкових діб загалом 18 із 365, тож у деяких порах їх зовсім мало.

In [26]:
# Q9. Як змінюється середній попит за діапазонами температури?
show("Q9", """
SELECT CASE WHEN temperature_c < 0  THEN '1: below 0'
            WHEN temperature_c < 10 THEN '2: 0-10'
            WHEN temperature_c < 20 THEN '3: 10-20'
            WHEN temperature_c < 30 THEN '4: 20-30'
            ELSE '5: 30+' END AS temp_band,
       COUNT(*) AS n_hours,
       ROUND(AVG(rented_bike_count), 1) AS avg_rentals
FROM hw3_bike_clean
WHERE is_functioning_day
GROUP BY temp_band
ORDER BY temp_band
""")

Q9


,temp_band,n_hours,avg_rentals
0,1: below 0,1433,198.0
1,2: 0-10,2118,426.3
2,3: 10-20,2050,837.2
3,4: 20-30,2340,1148.1
4,5: 30+,524,1112.2


Q9. Середній попит зростає з температурою: від 198,0 оренди за годину нижче 0 °C до 1148,1 у діапазоні 20-30 °C, тобто приблизно в 5,8 раза. Понад 30 °C попит не росте, а трохи падає (1112,2), тож залежність не монотонна: найкомфортніший діапазон 20-30 °C, а надмірна спека вже відлякує. Діапазон 30+ має лише 524 години проти 2340 у попередньому, тому його середнє менш стабільне. Для моделі це означає, що зв'язок температури з попитом нелінійний.

In [27]:
# Q10. Які 5 діб мали найбільше оренд за добу?
show("Q10", """
SELECT rent_date, SUM(rented_bike_count) AS total_rentals,
       ROUND(AVG(temperature_c), 1) AS avg_temp
FROM hw3_bike_clean
GROUP BY rent_date
ORDER BY total_rentals DESC, rent_date
LIMIT 5
""")

Q10


,rent_date,total_rentals,avg_temp
0,2018-06-13,36149,22.0
1,2018-06-19,35349,22.6
2,2018-06-08,35103,21.7
3,2018-06-20,34639,22.7
4,2018-06-21,34621,23.6


Q10. П'ять найактивніших діб усі припадають на червень 2018: від 34 621 до 36 149 оренд за добу, середня температура 21,7-23,6 °C. Максимум 13 червня (36 149) не збігається з добою рекордної години (19 червня, 3556 оренд о 18:00), бо рекорд доби складається з усіх 24 годин, а не з одного піку. Це показує, що попит залежить від комфортної температури протягом усього дня, а не лише від вечірнього піку. Для прогнозування корисні й добові агрегати, й погодинні.

Якість даних. Набір Seoul Bike повний: 8760 рядків (365 діб × 24 години), дублікатів за ключем (дата, година) немає, порожніх значень і маркерів ?/unknown не виявлено. Проблеми не в пропусках, а в кодуванні: дата записана текстом D/M/YYYY (формат довелося визначати за даними), двійкові ознаки словами (Holiday/No Holiday, Yes/No), а одиниці виміру, наприклад видимість у десятках метрів, відомі лише з документації. Розподіл цілі скошений: середнє 704,6 оренди за годину, максимум 3556.

Features. Використала б годину, пору року, святковий день, температуру, вологість, вітер, опади, а з дати місяць і день тижня. Точку роси не брала б: вона значною мірою залежить від температури й вологості. Технічний bike_row_id не несе змісту.

Leakage. Фактична погода тієї ж години стає відомою лише після неї: для прогнозу наперед потрібен прогноз погоди. Functioning Day майже повністю визначає ціль: у 295 нефункціональних годинах оренд 0, тому цю ознаку можна використовувати, лише якщо статус відомий заздалегідь.

Pipeline. Запропонувала б data engineer віддавати дату в ISO 8601, булеві поля як true/false, додати словник колонок з одиницями виміру та перевірку повноти (24 години × кількість днів).

ML-задача. Регресія: прогноз Rented Bike Count; з огляду на часову структуру також forecasting.